# Recomendador de popularidad por género (baseline, sin modelo)

El baseline más simple de los que hay en este repo: sin ML, sin similitud coseno entre géneros
(eso ya está en `recomendador_similitud_genero.ipynb`) — acá cada libro entra o no entra, no hay
ponderación entre géneros.

1. **Géneros leídos por cada lector**: el conjunto de géneros (canónicos) en los que dejó alguna
   calificación en `interacciones`.
2. **Score por libro**: promedio de calificaciones, solo para libros con **>= 3 calificaciones
   válidas** (mismo piso que el notebook de similitud, para no dejar que un libro con una sola
   review de 10 le gane a uno con miles de reviews en 9).
3. **Candidatos de un lector** = todos los libros cuyo género canónico está en el conjunto de
   géneros que ese lector leyó, sin importar de qué género sea cada libro individualmente (no se
   pondera por qué tan leído está ese género en particular — es lo que separa este notebook del de
   similitud coseno).
4. Se descartan los libros que el lector ya calificó.
5. Top 20 por `avg_score` descendente (desempate por `n_ratings` descendente, igual que en
   similitud de género).

**Nota sobre autores**: la consigna original pedía identificar también los autores leídos por cada
lector, pero la regla de ranking que describe solo usa género — no autor. Se optó por no usar autor
como filtro (habría vuelto el candidate pool demasiado angosto, o habría requerido una regla de
combinación no especificada).

In [1]:
import sqlite3

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DB_PATH = "datos/data.db"
EJEMPLO_CSV = "datos/ejemplo.csv"
GENERO_CANON_LOOKUP_CSV = "datos/genero_canon_lookup.csv"
OUTPUT_CSV = "datos/popularity_recommendation.csv"
RANDOM_STATE = 42
MIN_RATINGS_LIBRO = 3  # libros con menos calificaciones no se puntuan
TOP_N = 20

## 1. Carga y limpieza de `interacciones`

Mismo criterio de validación que en `recomendador_similitud_genero.ipynb`: se descarta la fila con
fecha corrupta y las interacciones sin metadata de libro o lector.

In [2]:
conn = sqlite3.connect(DB_PATH)
lectores = pd.read_sql_query("SELECT * FROM lectores", conn)
libros = pd.read_sql_query("SELECT * FROM libros", conn)
interacciones = pd.read_sql_query("SELECT * FROM interacciones", conn)
conn.close()

fecha_ok = interacciones["fecha"].str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
libro_ok = interacciones["id_libro"].isin(set(libros["id_libro"]))
lector_ok = interacciones["id_lector"].isin(set(lectores["id_lector"]))
validas = fecha_ok & libro_ok & lector_ok

inter = interacciones[validas].merge(libros[["id_libro", "genero"]], on="id_libro", how="left")
inter["rating"] = inter["rating"].astype(float)
print(f"interacciones validas: {len(inter):,} / {len(interacciones):,}")

interacciones validas: 461,073 / 461,408


## 2. Género canónico

Se reusa el lookup `datos/genero_canon_lookup.csv` generado en `dataset_features_genero.ipynb`
(mismos 53 géneros canónicos que usa `recomendador_similitud_genero.ipynb`), en vez de recalcular
el fuzzy matching acá — así los dos notebooks quedan consistentes entre sí.

In [3]:
genero_canon_lut = pd.read_csv(GENERO_CANON_LOOKUP_CSV)
genero_canon_map = dict(zip(genero_canon_lut["genero_original"], genero_canon_lut["genero_canonico"]))

inter["genero_norm"] = inter["genero"].str.strip().str.lower()
inter["genero_canon"] = inter["genero_norm"].map(genero_canon_map)

sin_mapear = inter.loc[inter["genero_norm"].notna() & inter["genero_canon"].isna(), "genero_norm"].unique()
print(f"generos canonicos: {inter['genero_canon'].nunique()}")
print(f"interacciones sin genero (libro sin genero en catalogo, excluidas de todo lo que sigue): "
      f"{inter['genero_canon'].isna().sum():,} ({inter['genero_canon'].isna().mean():.0%})")
if len(sin_mapear):
    print(f"AVISO: {len(sin_mapear)} generos no estan en el lookup y quedaron sin mapear: {sorted(sin_mapear)}")

inter_g = inter.dropna(subset=["genero_canon"]).copy()

generos canonicos: 52
interacciones sin genero (libro sin genero en catalogo, excluidas de todo lo que sigue): 1 (0%)
AVISO: 1 generos no estan en el lookup y quedaron sin mapear: ['']


## 3. Score de popularidad por libro (>= 3 calificaciones)

In [4]:
libro_stats = inter_g.groupby("id_libro").agg(
    genero_libro=("genero_canon", "first"), avg_score=("rating", "mean"), n_ratings=("rating", "size")
)
candidatos = libro_stats[libro_stats["n_ratings"] >= MIN_RATINGS_LIBRO].copy()
print(f"libros con genero: {len(libro_stats):,}  candidatos (>= {MIN_RATINGS_LIBRO} calificaciones): {len(candidatos):,}")
candidatos.sort_values(["avg_score", "n_ratings"], ascending=[False, False]).head()

libros con genero: 48,062  candidatos (>= 3 calificaciones): 18,706


,genero_libro,avg_score,n_ratings
id_libro,,,
el-aroma-de-los-mangos,ficción literaria,10.0,6
episodios-nacionales-quinta-serie,histórica y aventuras,10.0,5
maestros-antiguos-comedia,literatura contemporánea,10.0,5
promesas-de-sangre-las-lagrimas-de-kaiu-ii,"fantástica, ciencia ficción",10.0,5
atlas-de-islas-sin-coches,lecturas complementarias,10.0,4


## 4. Géneros leídos por cada lector

Conjunto de géneros canónicos en los que el lector dejó al menos una calificación (válida, con
libro identificado). No hay "género principal" acá como en el notebook de similitud — un lector
puede tener varios géneros leídos y todos cuentan igual para armar su candidate pool.

In [5]:
generos_leidos_por_lector = inter_g.groupby("id_lector")["genero_canon"].apply(set).to_dict()
print(f"lectores con al menos un genero leido: {len(generos_leidos_por_lector):,}")

n_generos = pd.Series({lid: len(gs) for lid, gs in generos_leidos_por_lector.items()})
print(n_generos.describe())

lectores con al menos un genero leido: 10,667
count    10667.000000
mean         5.697197
std          4.642679
min          1.000000
25%          2.000000
50%          4.000000
75%          9.000000
max         25.000000
dtype: float64


## 5. Top 20 por lector para `ejemplo.csv`

Como no hay ponderación por género (a diferencia del notebook de similitud), el ranking de
candidatos es el mismo para todos los lectores — un único orden global por `avg_score` (desempate
por `n_ratings`). Por lector, alcanza con filtrar ese ranking a los libros cuyo género está en su
propio conjunto de géneros leídos, sacar los que ya leyó, y tomar los primeros 20.

**Lectores sin ningún género leído** (sin calificaciones válidas, o todas en libros sin género):
no hay forma de calcular un candidate pool con la regla pedida, así que van a *fallback* al
ranking global (todos los candidatos, sin filtrar por género) — es la única forma de darles
igualmente una recomendación.

In [6]:
ranking_global = candidatos.sort_values(["avg_score", "n_ratings"], ascending=[False, False])

ejemplo = pd.read_csv(EJEMPLO_CSV)
lectores_objetivo = sorted(ejemplo["id_lector"].unique())
print(f"lectores objetivo: {len(lectores_objetivo)}")

leidos_por_lector = interacciones.groupby("id_lector")["id_libro"].apply(set).to_dict()

sin_genero_leido = sum(1 for l in lectores_objetivo if l not in generos_leidos_por_lector)
print(f"lectores objetivo sin ningun genero leido (fallback a ranking global): {sin_genero_leido}")

filas = []
for lid in lectores_objetivo:
    generos_lector = generos_leidos_por_lector.get(lid)
    leidos = leidos_por_lector.get(lid, set())

    if generos_lector:
        pool = ranking_global[ranking_global["genero_libro"].isin(generos_lector)]
    else:
        pool = ranking_global

    top20 = pool[~pool.index.isin(leidos)].head(TOP_N)
    for id_libro, fila in top20.iterrows():
        filas.append((lid, id_libro, fila["genero_libro"], fila["avg_score"], fila["n_ratings"]))

recomendaciones = pd.DataFrame(
    filas, columns=["id_lector", "id_libro", "genero_libro", "avg_score", "n_ratings"]
)
print(f"total: {recomendaciones.shape}")

lectores objetivo: 832


lectores objetivo sin ningun genero leido (fallback a ranking global): 17

total: (16640, 5)


## 6. Validaciones de sanidad

In [7]:
filas_por_lector = recomendaciones.groupby("id_lector").size()
print("lectores objetivo cubiertos:", filas_por_lector.index.isin(lectores_objetivo).all() and len(filas_por_lector) == len(lectores_objetivo))
print(filas_por_lector.value_counts())

incompletos = filas_por_lector[filas_por_lector < TOP_N]
if len(incompletos):
    print(f"\n{len(incompletos)} lectores con menos de {TOP_N} recomendaciones (candidate pool angosto):")
    print(incompletos)

dup = recomendaciones.duplicated(subset=["id_lector", "id_libro"]).sum()
ya_leidos_set = {(lid, lib) for lid, libs in leidos_por_lector.items() for lib in libs}
recomendados_set = set(map(tuple, recomendaciones[["id_lector", "id_libro"]].itertuples(index=False, name=None)))
interseccion = recomendados_set & ya_leidos_set

print(f"\npares duplicados: {dup}")
print(f"recomendaciones que ya estaban leidas (deberia ser 0): {len(interseccion)}")

assert dup == 0, "hay pares (lector, libro) duplicados"
assert len(interseccion) == 0, "se recomendo un libro ya leido"
print("\nOK: sin duplicados, sin libros ya leidos.")

lectores objetivo cubiertos: True
20    832
Name: count, dtype: int64

pares duplicados: 0
recomendaciones que ya estaban leidas (deberia ser 0): 0

OK: sin duplicados, sin libros ya leidos.


## 7. CSV final para Kaggle (misma estructura que `ejemplo.csv`)

In [8]:
salida = (
    recomendaciones.sort_values(["id_lector", "avg_score", "n_ratings"], ascending=[True, False, False])
    [["id_lector", "id_libro"]]
    .reset_index(drop=True)
)

print("columnas:", salida.columns.tolist(), "== ejemplo.csv:", salida.columns.tolist() == ejemplo.columns.tolist())
print("filas:", len(salida), " (ejemplo.csv tiene", len(ejemplo), ")")

salida.to_csv(OUTPUT_CSV, index=False)
print(f"\nguardado en {OUTPUT_CSV}")
salida.head(10)

columnas: ['id_lector', 'id_libro'] == ejemplo.csv: True
filas: 16640  (ejemplo.csv tiene 16640 )

guardado en datos/popularity_recommendation.csv


,id_lector,id_libro
0,05-03-1970,el-aroma-de-los-mangos
1,05-03-1970,episodios-nacionales-quinta-serie
2,05-03-1970,maestros-antiguos-comedia
3,05-03-1970,promesas-de-sangre-las-lagrimas-de-kaiu-ii
4,05-03-1970,atlas-de-islas-sin-coches
5,05-03-1970,el-eco-de-los-disparos-cultura-y-memoria-de-la...
6,05-03-1970,evocacion-la-historia-de-marilia
7,05-03-1970,infierno
8,05-03-1970,las-lagrimas-de-kaiu
9,05-03-1970,novelas-2


## 8. Ejemplo

In [9]:
muestra = recomendaciones["id_lector"].drop_duplicates().sample(2, random_state=RANDOM_STATE).tolist()
for lid in muestra:
    generos_lector = generos_leidos_por_lector.get(lid, "(sin genero leido -> fallback global)")
    top5 = recomendaciones[recomendaciones["id_lector"] == lid].sort_values("avg_score", ascending=False).head(5)
    top5 = top5.merge(libros[["id_libro", "titulo", "autor", "genero"]], on="id_libro", how="left")
    print(f"\n=== top 5 para {lid} (generos leidos: {generos_lector}) ===")
    print(top5[["id_libro", "titulo", "autor", "genero", "avg_score", "n_ratings"]].to_string(index=False))


=== top 5 para omallorqui (generos leidos: {'ficción literaria', 'fantástica, ciencia ficción', 'narrativa', 'histórica y aventuras', 'biografías, memorias', 'literatura contemporánea', 'infantil y juvenil', 'lecturas complementarias', 'humor', 'ensayo', 'clásicos de la literatura', 'novela negra, intriga, terror', 'poesía, teatro', 'no ficción'}) ===
                         id_libro                             titulo                   autor                        genero  avg_score  n_ratings
           el-aroma-de-los-mangos             EL AROMA DE LOS MANGOS             AKAM, PAULO             Ficción literaria       10.0          6
episodios-nacionales-quinta-serie EPISODIOS NACIONALES. QUINTA SERIE    PÉREZ GALDÓS, BENITO         Histórica y aventuras       10.0          5
             el-hombre-de-damasco               EL HOMBRE DE DAMASCO NÚÑEZ ALONSO, ALEJANDRO         Histórica y aventuras       10.0          3
      el-dulce-amargor-del-crimen        EL DULCE AMARGOR DEL CRI

**Resumen**: baseline de popularidad por género — para cada lector, universo de candidatos = todos
los libros (>= 3 calificaciones) cuyo género está entre los géneros que ese lector calificó alguna
vez, ordenados por rating promedio descendente (desempate por cantidad de reviews). Sin ponderación
entre géneros ni modelo. Libros ya leídos excluidos. Lectores sin ningún género leído usan el
ranking global de popularidad como fallback. Resultado en `datos/popularity_recommendation.csv`,
misma estructura que `ejemplo.csv`.